# 🧠 Notebook 2: ORNJ Concept & Term Extraction

Extract clinical concepts from ORNJ texts: anatomical structures, clinical findings, treatments, classification terms.

---

In [ ]:
!pip install -q PyMuPDF pdfplumber pytesseract Pillow spacy scikit-learn owlready2 rdflib networkx matplotlib anthropic tqdm
!apt-get install -q tesseract-ocr > /dev/null 2>&1
!python -m spacy download en_core_web_sm -q
!git clone https://github.com/YOUR_USERNAME/orn-ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'orn-ontology-builder')

## 1. Load Text

In [ ]:
import json, os
if os.path.exists('output/extracted_texts.json'):
    with open('output/extracted_texts.json') as f:
        texts = [d['full_text'] for d in json.load(f)]
    print(f'Loaded {len(texts)} documents')
else:
# Demo text: ORNJ classification systems (use when no PDFs uploaded)
demo_texts = [
    """Notani et al. divided cases of mandibular osteoradionecrosis into
    three grades based on the extent of the lesion. Grade I is defined
    as ORN confined to the alveolar bone. Grade II is defined as ORN
    limited to the alveolar bone and/or the mandible above the level
    of the inferior alveolar canal. Grade III is ORN extending below
    the inferior alveolar canal with or without pathological fracture
    or orocutaneous fistula. Treatments such as sequestrectomy,
    marginal resection, and segmental resection with free flap
    reconstruction are recommended based on grade severity.""",

    """The ClinRad classification system proposed by Watson et al. in 2024
    considers both clinical and radiographic features to stage ORNJ.
    Grade 0 represents minor bone spicules without clinical significance.
    Grade 1 involves alveolar bone necrosis detected radiographically
    without bone exposure. Grade 2 involves alveolar bone necrosis with
    exposed bone or probe-to-bone positive findings. Grade 3 involves
    basilar bone involvement or maxillary sinus involvement. Grade 4
    involves pathological fracture, orocutaneous fistula, or extensive
    bone destruction. The ClinRad system was endorsed by ISOO-MASCC-ASCO.""",

    """Marx proposed a three-stage system for osteoradionecrosis in 1983.
    Stage I patients exhibit exposed bone that has failed to heal for
    at least 6 months and receive 30 sessions of hyperbaric oxygen
    therapy. Stage I responders show tissue softening and spontaneous
    sequestration. Stage II involves non-responsive cases requiring
    surgery combined with HBO. Stage III involves pathological fracture
    or orocutaneous fistula requiring mandibular resection and free flap
    reconstruction. Risk factors include radiation dose exceeding 60 Gy,
    smoking, dental extraction after radiotherapy, and stage III-IV
    periodontal disease.""",
]
    print(f'Using {len(demo_texts)} demo documents')
    texts = demo_texts

## 2. Preprocess

In [ ]:
from src.preprocessor import TextPreprocessor
pp = TextPreprocessor(min_segment_length=15)
segments = []
for i, t in enumerate(texts):
    segments.extend(pp.preprocess(t, source_doc=f'doc_{i}'))
print(f'Total segments: {len(segments)}')

## 3. Named Entity Recognition

In [ ]:
import spacy
from collections import Counter
nlp = spacy.load('en_core_web_sm')
full_text = ' '.join(s.text for s in segments)
doc = nlp(full_text)
entity_counter = Counter()
for ent in doc.ents:
    entity_counter[(ent.text, ent.label_)] += 1
print('Top Named Entities:')
for (text, label), count in entity_counter.most_common(20):
    print(f'  {text:<35} {label:<10} {count}')

## 4. Full Concept Extraction

In [ ]:
from src.concept_extractor import ConceptExtractor
extractor = ConceptExtractor(min_frequency=1, max_concepts=200)
concepts = extractor.extract(segments)
print(f'\nExtracted {len(concepts)} concepts:')
for c in concepts[:25]:
    print(f'  {c.name:<35} {c.label:<35} freq={c.frequency}')

## 5. Save

In [ ]:
os.makedirs('output', exist_ok=True)
with open('output/concepts.json', 'w') as f:
    json.dump([{'name': c.name, 'label': c.label, 'frequency': c.frequency} for c in concepts], f, indent=2)
print(f'Saved {len(concepts)} concepts')

---
**Next:** [Notebook 3 — Relation Extraction](03_relation_extraction.ipynb)